# Credit Card Fraud Detection: Exploratory Data Analysis

This notebook provides a concise exploratory analysis of the Kaggle credit card fraud dataset.

Goals:
- inspect dataset structure
- quantify class imbalance
- examine transaction amount behavior
- visualize a feature correlation pattern
- summarize implications for modeling

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'creditcard.csv'

df = pd.read_csv(DATA_PATH)
df.head()

## 1. Dataset Overview

In [ ]:
print(f'Shape: {df.shape}')
print('\nColumns:')
print(df.columns.tolist())
print('\nMissing values per column:')
print(df.isna().sum().sort_values(ascending=False).head())
print('\nData types:')
print(df.dtypes.head())

## 2. Class Distribution

The core challenge in this dataset is extreme class imbalance.

In [ ]:
class_counts = df['Class'].value_counts().sort_index()
class_ratio = df['Class'].value_counts(normalize=True).sort_index() * 100

summary = pd.DataFrame({
    'count': class_counts,
    'percentage': class_ratio.round(4)
})
summary.index = ['Non-Fraud (0)', 'Fraud (1)']
summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Class')
plt.title('Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

## 3. Transaction Amount Analysis

Although most PCA-based features are anonymized, `Amount` remains interpretable and useful for inspection.

In [ ]:
df['Amount'].describe().to_frame().T

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Amount'], bins=60, kde=True)
plt.title('Overall Transaction Amount Distribution')
plt.xlabel('Amount')
plt.ylabel('Frequency')
plt.xlim(0, df['Amount'].quantile(0.99))
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(data=df, x='Class', y='Amount')
plt.title('Transaction Amount by Class')
plt.xlabel('Class')
plt.ylabel('Amount')
plt.ylim(0, df['Amount'].quantile(0.99))
plt.show()

## 4. Feature Correlation Snapshot

The full feature space is high-dimensional. For readability, this notebook shows a smaller correlation heatmap using a subset of features plus `Amount` and `Class`.

In [ ]:
selected_columns = ['V1', 'V2', 'V3', 'V4', 'V10', 'V12', 'V14', 'V17', 'Amount', 'Class']
corr = df[selected_columns].corr()

plt.figure(figsize=(9, 6))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Correlation Heatmap (Selected Features)')
plt.show()

## 5. Practical Takeaways

- The fraud class is extremely rare, so accuracy is not an informative primary metric.
- Precision-recall tradeoffs are central to the modeling objective.
- Threshold tuning is necessary because the default decision boundary may not align with business goals.
- `Amount` has a skewed distribution and benefits from scaling before training certain models.
- Validation `PR-AUC`, fraud recall, and fraud F1-score are better indicators than overall accuracy for this task.